In [ ]:
# import pandas as pd
# import os
#
# def read_and_convert_to_mt_bert_df(csv_file:str):
#     df = pd.read_csv(csv_file)
#     df = df[["repository", "text", "label"]] # to single line
#     df.rename(columns={"repository": "projectname", "text": "Abstract", "label": "original_label"}, inplace=True)
#     return df
#
# maldonado_train_comment = open('../config/baseline/origin/data--train.txt', 'r').readlines()
# maldonado_train_label = open('../config/baseline/origin/label--train.txt', 'r').readlines()
# maldonado_train_comment = list(map(lambda x: x.strip(), maldonado_train_comment))
# maldonado_train_label = list(map(lambda x: x.strip(), maldonado_train_label))
# maldonado_train_label = list(map(lambda x : "yes" if x.lower() == 'positive' else "no", maldonado_train_label))
# guo_df = pd.DataFrame(data = {
#     "projectname": ["all_projects"]* len(maldonado_train_comment),
#     "Abstract": maldonado_train_comment,
#     "original_label": maldonado_train_label,
#     "datasetname": ["guo"]* len(maldonado_train_comment),
# })
# MT_BERT_FILE_FORMAT = "../cache/baseline/mt_bert/satd/multi_train/{}_data/{}.csv"
# for ds in ["guo_duplicate", "guo_unique", "our_duplicate", "our_unique"]:
#     train_file = MT_BERT_FILE_FORMAT.format(ds, f"{ds}_code_comments_train")
#     test_file = MT_BERT_FILE_FORMAT.format(ds, f"{ds}_code_comments_test")
#     for file in [train_file, test_file]:
#         os.makedirs(os.path.dirname(file), exist_ok=True)
#
#     if "guo" in ds:
#         guo_df.to_csv(train_file, index=False)
#
#     if "duplicate" in ds:
#         if "guo" not in ds:
#             read_and_convert_to_mt_bert_df("../data/duplicate_detect_train.csv").to_csv(train_file, index=False)
#         read_and_convert_to_mt_bert_df("../data/duplicate_detect_test.csv").to_csv(test_file, index=False)
#     if "unique" in ds:
#         if "guo" not in ds:
#             read_and_convert_to_mt_bert_df("../data/unique_detect_train.csv").to_csv(train_file, index=False)
#         read_and_convert_to_mt_bert_df("../data/unique_detect_test.csv").to_csv(test_file, index=False)
#     test_df = pd.read_csv(file)
#     for dataset_type in ["train", "test"]:
#         for source in ["commit", "issue", "pr"]:
#             other_file = MT_BERT_FILE_FORMAT.format(ds, f"{ds}_{source}_{dataset_type}")
#             os.makedirs(os.path.dirname(other_file), exist_ok=True)
#             test_df[:10].to_csv(other_file, index=False)
#
#


In [ ]:
# Prepare Training and Test Dataset for MT-BERT
import pandas as pd
import os
from sklearn.model_selection import StratifiedGroupKFold
import pandas as pd

BASE_BERT_DIRECTORY = "../cache/baseline/bert"
def convert_to_mt_bert_df(df: pd.DataFrame):
    """
    Converts dataframe to MT BERT format
    :param df: Technical Debt formatted dataframe
    :return: Mt BERT formatted dataframe
    """
    df = df[["repository", "text", "label"]] # to single line
    return df.rename(columns={"repository": "projectname", "text": "Abstract", "label": "original_label"})

def init_bert_data_directory(variant_name:str, training_type:str, dataset_name:str, train_df:pd.DataFrame, test_df:pd.DataFrame):
    """
    :param variant_name: Name of training variant
    :param training_type: Type of training variant(trained means technical debt training data pretrained means with other data)
    :param dataset_name: Unique or duplicate dataset name
    :param train_df: Training dataframe
    :param test_df: Testing dataframe
    """
    # Template for create 4 different types of training and test csv dataset files
    MT_BERT_FILE_FORMAT = BASE_BERT_DIRECTORY + "/input/satd/multi_train/{}_data/{}.csv"
    # All the relevant concatenated as prefix for later splitting and recovery
    # trained_bert_default_data
    # trained_unique_bert_5fcv1
    file_name_prefix = f"{training_type}_{dataset_name}_bert_{variant_name}"
    train_file = MT_BERT_FILE_FORMAT.format(file_name_prefix, f"{file_name_prefix}_code_comments_train")
    test_file = MT_BERT_FILE_FORMAT.format(file_name_prefix, f"{file_name_prefix}_code_comments_test")

    # Optionally save test dataset for post training testing
    post_train_test_file = f"{BASE_BERT_DIRECTORY}/input/unclassified_files/{file_name_prefix}_code_comments_test.csv"
    os.makedirs(os.path.dirname(post_train_test_file), exist_ok=True)
    convert_to_mt_bert_df(test_df)[["Abstract"]].to_csv(post_train_test_file, index=False)


    # Create the real train and test file for code comment
    for file in [train_file, test_file]:
        os.makedirs(os.path.dirname(file), exist_ok=True)
    convert_to_mt_bert_df(train_df).to_csv(train_file, index=False)
    convert_to_mt_bert_df(test_df).to_csv(test_file, index=False)

    # Create dummy train and test files for the unused other three types as model input is mandatory
    for dataset_type in ["train", "test"]:
        for source in ["commit", "issue", "pr"]:
            other_file = MT_BERT_FILE_FORMAT.format(file_name_prefix, f"{file_name_prefix}_{source}_{dataset_type}")
            os.makedirs(os.path.dirname(other_file), exist_ok=True)
            convert_to_mt_bert_df(test_df)[:10].to_csv(other_file, index=False)

    # Command that will be executed for this training
    train_cmd = f"!{{sys.executable}} mt-bert-satd/run_mt-bert-satd-code-comment.py --data_dir {file_name_prefix} --output_dir model/{file_name_prefix}"
    # command that will be executed for thest testing
    test_cmd = f"!{{sys.executable}} mt-bert-satd/predict.py --task 4 --data_dir {file_name_prefix}_code_comments_test --output_dir output/{variant_name}/{training_type}/{dataset_name}/{dataset_name}"

    print(train_cmd)
    print(test_cmd)
    return file_name_prefix

# print("import sys")
# prepare training and test dataset for both unique and deuplicate dataset
for dataset_name in ["unique", "duplicate"]:
    full_train_df = pd.read_csv(f'../data/{dataset_name}_detect_train.csv')
    full_test_df = pd.read_csv(f'../data/{dataset_name}_detect_test.csv')
    # init training and testing dataset for default(without cross validation) experiment
    variant_directory = init_bert_data_directory("default", "trained", dataset_name, full_train_df, full_test_df)

    df = pd.concat([full_train_df, full_test_df])

    X = df["text"]
    y = df["label"]
    groups = df["repository"]

    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
    for fold, (train_idx, test_idx) in enumerate(cv.split(X, y, groups)):
        train_df = df.iloc[train_idx]
        test_df = df.iloc[test_idx]
        fold_suffix = f"5fcv-{fold+1}"
        # print(f'Fold fold_suffix: {len(train_df)} train and {len(test_df)} test samples')
        # init training and testing dataset for 5 fold cross validation experiment
        variant_directory = init_bert_data_directory(fold_suffix, "trained", dataset_name, train_df, test_df)

    # Create test files for pretrained model
    post_train_test_file = f"{BASE_BERT_DIRECTORY}/input/unclassified_files/{dataset_name}_code_comments_test.csv"
    os.makedirs(os.path.dirname(post_train_test_file), exist_ok=True)
    convert_to_mt_bert_df(full_test_df)[["Abstract"]].to_csv(post_train_test_file, index=False)





In [ ]:
import sys

py = sys.executable
folds = ["default", "5fcv-1", "5fcv-2", "5fcv-3", "5fcv-4", "5fcv-5"]
kinds = ["unique", "duplicate"]

for kind in kinds:
    for fold in folds:
        train_dir = f"trained_{kind}_bert_{fold}"
        test_dir  = f"{train_dir}_code_comments_test"

        !{py} mt-bert-satd/run_mt-bert-satd.py --data_dir {train_dir} --output_dir model/{train_dir}
        # No need to run prediction on newly trained model as after each iteration of the training the model runs test and we can simply take the last iteration's result.
        # !{py} mt-bert-satd/predict.py --task 4 --data_dir {test_dir} --output_dir output/{fold}/trained/{kind}/{kind}

In [ ]:
import sys
!{sys.executable} mt-bert-satd/predict.py --task 4 --data_dir unique_code_comments_test --output_dir predict_files/pretrained_unique_bert_default
# !{sys.executable} mt-bert-satd/predict.py --task 4 --data_dir duplicate_code_comments_test --output_dir predict_files/pretrained_duplicate_bert_default


In [ ]:
from dotenv import load_dotenv
import os
from Model import Model
from constant import *
from SimpleOutputLabelConverter import SimpleOutputLabelConverter
load_dotenv()

simple_output_label_converter = SimpleOutputLabelConverter({'yes', 'no'}, DEFAULT_DETECTION_CLASS)

detect_duplicate_test_df = pd.read_csv(f'../data/duplicate_detect_test.csv')
detect_duplicate_test_dataset = Dataset.from_pandas(detect_duplicate_test_df)

detect_unique_test_df = pd.read_csv(f'../data/unique_detect_test.csv')
detect_unique_test_dataset = Dataset.from_pandas(detect_unique_test_df)
for exp, file_name in [("guo_duplicate", "results_code_comments_5.txt"),
                       ("guo_unique", "results_code_comments_1.txt"),
                       ("our_duplicate", "results_code_comments_7.txt"),
                       ("our_unique", "results_code_comments_7.txt"),
                       ("our_duplicate_code_comments_test", "results_our_duplicate_code_comments_test4.txt"),
                       ("our_unique_code_comments_test", "results_our_unique_code_comments_test4.txt"),]:
    output_file = "../cache/baseline/mt_bert/model/{}/{}".format(exp, file_name)
    if os.path.exists(output_file):
        dataset_name = "duplicate" if "duplicate" in exp else "unique"
        dataset = detect_duplicate_test_dataset if "duplicate" == dataset_name else detect_unique_test_dataset
        model = Model('detect', f'bert-{exp}', simple_output_label_converter, 10_000)
        # model.fit(detect_train_dataset)

        raw_predicted_labels = open(output_file, 'r').readlines()
        raw_predicted_labels = list(map(lambda x: x.strip(), raw_predicted_labels))
        predicted_labels = list(map(lambda x: "yes" if x == "1" else "no", raw_predicted_labels))
        model.predict_end(dataset, dataset_name, predicted_labels, raw_predicted_labels)


In [ ]:
from dotenv import load_dotenv
import os
from Model import Model
from constant import *
from SimpleOutputLabelConverter import SimpleOutputLabelConverter
load_dotenv()

simple_output_label_converter = SimpleOutputLabelConverter({'yes', 'no'}, DEFAULT_DETECTION_CLASS)

detect_duplicate_test_df = pd.read_csv(f'../data/duplicate_detect_test.csv')
detect_duplicate_test_dataset = Dataset.from_pandas(detect_duplicate_test_df)

detect_unique_test_df = pd.read_csv(f'../data/unique_detect_test.csv')
detect_unique_test_dataset = Dataset.from_pandas(detect_unique_test_df)
for exp, file_name in [("guo_duplicate", "results_code_comments_5.txt"),
                       ("guo_unique", "results_code_comments_1.txt"),
                       ("our_duplicate", "results_code_comments_7.txt"),
                       ("our_unique", "results_code_comments_7.txt"),
                       ("our_duplicate_code_comments_test", "results_our_duplicate_code_comments_test4.txt"),
                       ("our_unique_code_comments_test", "results_our_unique_code_comments_test4.txt"),]:
    output_file = "../cache/baseline/mt_bert/model/{}/{}".format(exp, file_name)
    if os.path.exists(output_file):
        dataset_name = "duplicate" if "duplicate" in exp else "unique"
        dataset = detect_duplicate_test_dataset if "duplicate" == dataset_name else detect_unique_test_dataset
        model = Model('detect', f'bert-{exp}', simple_output_label_converter, 10_000)
        # model.fit(detect_train_dataset)

        raw_predicted_labels = open(output_file, 'r').readlines()
        raw_predicted_labels = list(map(lambda x: x.strip(), raw_predicted_labels))
        predicted_labels = list(map(lambda x: "yes" if x == "1" else "no", raw_predicted_labels))
        model.predict_end(dataset, dataset_name, predicted_labels, raw_predicted_labels)
